# 10 · Linear regression baseline

**Goal and hypothesis.** Fit a least-squares regressor to the binary label as a deliberately naive baseline. Its continuous score supports ROC-AUC and a validation-selected operating threshold; predictions are not probabilities and may fall outside `[0, 1]`.

This notebook reads the frozen **train** and **validation** tabular archives only. It does not open validation/test audio, mel caches, or any test manifest/cache. Notebook 90 is the only place permitted to score test.

In [1]:
from pathlib import Path
import sys, json, time, warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from dronedet.config import load_project_config
from dronedet.features import load_tabular
from dronedet.metrics import classification_metrics, select_f1_threshold, miss_rate_at_false_alarm_rate

cfg = load_project_config(ROOT / 'config.yaml')
P = cfg['P']
print('Project:', ROOT)
print('Representation: 84 train-centred-window features')

Project: D:\Work\ML Course Project\Functional Project
Representation: 84 train-centred-window features


D:\Work\ML Course Project\Functional Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Frozen inputs and scaler policy

Q-A is set to maximize validation F1, with ties resolved to the higher threshold; ROC-AUC and miss rates at 1%/5% false-alarm caps are also reported. Q-D is set to fit the tabular scaler on train rows excluding `silent` and `near_silent` only. The stored `crop_all_zero` flag is distinct in this cache; clips that are `crop_all_zero` but neither `silent` nor `near_silent` remain in this scaler fit, following the owner's explicit Q-D choice.

In [2]:
train = load_tabular(P, 'train')
val = load_tabular(P, 'val')
X_train = np.asarray(train['X'], dtype=np.float64)
y_train = np.asarray(train['y'], dtype=np.int8)
X_val = np.asarray(val['X'], dtype=np.float64)
y_val = np.asarray(val['y'], dtype=np.int8)
feature_names = np.asarray(train['feature_names']).astype(str)

assert X_train.shape == (7607, 84) and X_val.shape == (1510, 84)
assert np.isfinite(X_train).all() and np.isfinite(X_val).all()
assert np.array_equal(train['clip_id'], np.asarray(pd.read_csv(P['splits'] / 'train.csv', keep_default_na=False).clip_id))
assert np.array_equal(val['clip_id'], np.asarray(pd.read_csv(P['splits'] / 'val.csv', keep_default_na=False).clip_id))

stats_exclude = np.asarray(train['silent'], bool) | np.asarray(train['near_silent'], bool)
fit_rows = ~stats_exclude
assert int(fit_rows.sum()) == 7557
scaler = StandardScaler().fit(X_train[fit_rows])
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)

print('Rows:', {'train': len(y_train), 'validation': len(y_val)})
print('Scaler fit rows:', int(fit_rows.sum()), '| excluded silent/near-silent:', int(stats_exclude.sum()))
print('Train class counts:', dict(zip(*np.unique(y_train, return_counts=True))))
print('Balanced sample-weight range:', float(sample_weight.min()), float(sample_weight.max()))

Rows: {'train': 7607, 'validation': 1510}
Scaler fit rows: 7557 | excluded silent/near-silent: 50
Train class counts: {np.int8(0): np.int64(6640), np.int8(1): np.int64(967)}
Balanced sample-weight range: 0.572816265060241 3.9332988624612204


## Baseline and simple-regression illustration

The multiple regression is the model frozen by this notebook. The one-feature illustration chooses the feature with largest absolute Pearson correlation with the label on train only. Both fits use balanced sample weights. The scaler is fitted on the Q-D-eligible train rows and then applied to all train and validation rows.

In [3]:
# Main 84-feature least-squares model; no hyperparameter grid.
model = LinearRegression()
t0 = time.perf_counter()
model.fit(X_train_scaled, y_train, sample_weight=sample_weight)
train_time_s = time.perf_counter() - t0
train_scores = model.predict(X_train_scaled)
val_scores = model.predict(X_val_scaled)
condition_number = float(np.linalg.cond(X_train_scaled[fit_rows]))

# Train-only, one-feature comparison for illustration.
corrs = np.asarray([np.corrcoef(X_train[:, j], y_train)[0, 1] for j in range(X_train.shape[1])])
corrs = np.nan_to_num(corrs, nan=0.0, posinf=0.0, neginf=0.0)
simple_idx = int(np.argmax(np.abs(corrs)))
simple_scaler = StandardScaler().fit(X_train[fit_rows][:, [simple_idx]])
simple_model = LinearRegression()
simple_model.fit(simple_scaler.transform(X_train[:, [simple_idx]]), y_train, sample_weight=sample_weight)
simple_val_scores = simple_model.predict(simple_scaler.transform(X_val[:, [simple_idx]]))

print('Condition number of eligible scaled train design:', condition_number)
print('Simple-regression feature:', feature_names[simple_idx], '| train correlation:', float(corrs[simple_idx]))
print('Linear model parameters including intercept:', int(model.coef_.size + 1))

Condition number of eligible scaled train design: 47.589791454566964
Simple-regression feature: mfcc16_mean | train correlation: 0.37606257680997973
Linear model parameters including intercept: 85


## Validation selection and operating points

Threshold selection uses validation scores only. The 0.5 threshold is shown as the naive reference, then the Q-A validation-F1 threshold is reported. The 1% and 5% false-alarm operating points maximize recall subject to each validation false-alarm cap.

In [4]:
threshold = select_f1_threshold(y_val, val_scores)
metrics_f1 = classification_metrics(y_val, val_scores, threshold=threshold)
metrics_half = classification_metrics(y_val, val_scores, threshold=0.5)
miss_1, threshold_far_1 = miss_rate_at_false_alarm_rate(y_val, val_scores, max_false_alarm_rate=0.01)
miss_5, threshold_far_5 = miss_rate_at_false_alarm_rate(y_val, val_scores, max_false_alarm_rate=0.05)

simple_threshold = select_f1_threshold(y_val, simple_val_scores)
simple_metrics = classification_metrics(y_val, simple_val_scores, threshold=simple_threshold)
summary = pd.DataFrame([
    {'model':'multiple_linear_regression','threshold_0_5':0.5,'f1_at_0_5':metrics_half['f1'],
     'selected_threshold':threshold,'val_f1':metrics_f1['f1'],'val_roc_auc':metrics_f1['roc_auc'],
     'false_alarm_rate':metrics_f1['false_alarm_rate'],'miss_rate':metrics_f1['miss_rate'],
     'miss_rate_at_far_1pct':miss_1,'threshold_at_far_1pct':threshold_far_1,
     'miss_rate_at_far_5pct':miss_5,'threshold_at_far_5pct':threshold_far_5},
    {'model':'simple_linear_regression','threshold_0_5':0.5,
     'f1_at_0_5':classification_metrics(y_val,simple_val_scores,threshold=0.5)['f1'],
     'selected_threshold':simple_threshold,'val_f1':simple_metrics['f1'],'val_roc_auc':simple_metrics['roc_auc'],
     'false_alarm_rate':simple_metrics['false_alarm_rate'],'miss_rate':simple_metrics['miss_rate'],
     'miss_rate_at_far_1pct':miss_rate_at_false_alarm_rate(y_val,simple_val_scores,max_false_alarm_rate=0.01)[0],
     'threshold_at_far_1pct':miss_rate_at_false_alarm_rate(y_val,simple_val_scores,max_false_alarm_rate=0.01)[1],
     'miss_rate_at_far_5pct':miss_rate_at_false_alarm_rate(y_val,simple_val_scores,max_false_alarm_rate=0.05)[0],
     'threshold_at_far_5pct':miss_rate_at_false_alarm_rate(y_val,simple_val_scores,max_false_alarm_rate=0.05)[1]},
])
display(summary)

,model,threshold_0_5,f1_at_0_5,selected_threshold,val_f1,val_roc_auc,false_alarm_rate,miss_rate,miss_rate_at_far_1pct,threshold_at_far_1pct,miss_rate_at_far_5pct,threshold_at_far_5pct
0,multiple_linear_regression,0.5,0.644401,0.66205,0.727273,0.953436,0.040000,0.264865,0.572973,0.900846,0.243243,0.644090
1,simple_linear_regression,0.5,0.368000,0.55937,0.386598,0.731219,0.096604,0.594595,0.870270,0.955722,0.697297,0.654687


## Diagnostics

These plots and errors use validation predictions; the residual plot and coefficients use train data. No audio is loaded or described.

In [5]:
FIG = ROOT / 'results' / 'figures'
TABLES = ROOT / 'results' / 'tables'
for path in (FIG, TABLES): path.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))
for label, name in ((0, 'Unknown'), (1, 'Drone')):
    ax.hist(val_scores[y_val == label], bins=35, alpha=0.55, label=name)
ax.axvline(0.5, color='black', linestyle=':', label='Threshold 0.5')
ax.axvline(threshold, color='red', linestyle='--', label='Validation-F1 threshold')
ax.set(xlabel='Linear-regression score', ylabel='Validation clips')
ax.legend(); fig.tight_layout()
fig.savefig(FIG / '10_linear_regression_validation_scores.png', dpi=160); plt.close(fig)

coef = pd.DataFrame({'feature':feature_names,'coefficient':model.coef_})
coef['abs_coefficient'] = coef.coefficient.abs()
coef.sort_values('abs_coefficient', ascending=False).to_csv(TABLES / '10_linear_regression_coefficients.csv', index=False)
top = coef.nlargest(15, 'abs_coefficient').sort_values('coefficient')
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top.feature, top.coefficient)
ax.set(xlabel='Coefficient on standardized feature', ylabel='Feature')
fig.tight_layout(); fig.savefig(FIG / '10_linear_regression_coefficients.png', dpi=160); plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(train_scores, y_train - train_scores, s=10, alpha=0.25)
ax.axhline(0, color='black', linewidth=1)
ax.set(xlabel='Fitted train score', ylabel='Train residual (label − fitted score)')
fig.tight_layout(); fig.savefig(FIG / '10_linear_regression_residuals.png', dpi=160); plt.close(fig)

val_pred = val_scores >= threshold
mis = pd.DataFrame({'clip_id':val['clip_id'].astype(str),'group':val['group'].astype(str),
                    'y_true':y_val,'score':val_scores,'y_pred':val_pred.astype(np.int8)})
mis = mis.loc[mis.y_true != mis.y_pred].copy()
mis['error_margin'] = np.where(mis.y_true.eq(1), threshold - mis.score, mis.score - threshold)
mis.sort_values('error_margin', ascending=False).to_csv(TABLES / '10_linear_regression_validation_errors.csv', index=False)
print('Validation errors:',len(mis),'| examples:')
display(mis.head(10))
print('Model scores were inspected for validation only.')

Validation errors: 102 | examples:


,clip_id,group,y_true,score,y_pred,error_margin
36,unknown/1-17742-A-121.wav,1-17742,0,0.725262,1,0.063212
50,unknown/1-19872-B-360.wav,1-19872,0,0.805306,1,0.143256
51,unknown/1-19872-B-361.wav,1-19872,0,0.817683,1,0.155634
52,unknown/1-19872-B-362.wav,1-19872,0,0.754011,1,0.091962
53,unknown/1-19872-B-363.wav,1-19872,0,0.900025,1,0.237975
54,unknown/1-19872-B-364.wav,1-19872,0,0.797971,1,0.135921
80,unknown/1-24524-C-190.wav,1-24524,0,0.674015,1,0.011965
105,unknown/1-39937-A-280.wav,1-39937,0,0.685705,1,0.023655
106,unknown/1-39937-A-281.wav,1-39937,0,1.057950,1,0.395901
108,unknown/1-39937-A-283.wav,1-39937,0,0.723956,1,0.061906


Model scores were inspected for validation only.


## Freeze and verdict-card draft

Freeze the multiple-regression pipeline and its validation-selected threshold. The one-feature fit is a comparison only. The artifact contains the fitted train scaler, the estimator, feature order, exclusion policy, threshold and front-end hash. No test file is opened.

In [6]:
from sklearn.pipeline import Pipeline
import hashlib

frozen_pipeline = Pipeline([('scaler', scaler), ('linear_regression', model)])
MODEL_DIR = ROOT / 'results' / 'models'
CARD_DIR = ROOT / 'results' / 'cards'
MODEL_DIR.mkdir(parents=True, exist_ok=True); CARD_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / '10_linear_regression.joblib'
joblib.dump({'pipeline':frozen_pipeline,'feature_names':feature_names.tolist(),
             'threshold':float(threshold),'threshold_rule':'validation_f1_max_ties_higher',
             'scaler_fit_exclusion':'silent OR near_silent; crop_all_zero alone is not excluded',
             'front_end_hash':json.loads((P['spec'] / 'frontend_hash.json').read_text(encoding='utf-8'))['params_hash']}, model_path)
summary.to_csv(TABLES / '10_linear_regression_validation_metrics.csv', index=False)

front_end_hash = json.loads((P['spec'] / 'frontend_hash.json').read_text(encoding='utf-8'))['params_hash']
card = {
 'model':'linear_regression','representation':'84 tabular features',
 'best_hyperparameters':{},
 'val_metrics':{**metrics_f1,'threshold':float(threshold),'metrics_at_0_5':metrics_half,
                'miss_rate_at_far_1pct':float(miss_1),'threshold_at_far_1pct':float(threshold_far_1),
                'miss_rate_at_far_5pct':float(miss_5),'threshold_at_far_5pct':float(threshold_far_5)},
 'seeds':[int(cfg['seed'])], 'train_time_s':float(train_time_s),
 'strengths':['simple deterministic linear baseline','fast fit and compact 85-parameter model'],
 'weaknesses':['least-squares outputs are not probabilities and are unbounded',
               'correlated descriptors can make coefficients unstable'],
 'failure_cases_noted':mis.head(10)[['clip_id','group','y_true','score','y_pred']].to_dict(orient='records'),
 'decision_log_ids':['D038','D044','D045','D047','D053'],
 'threshold':float(threshold),'threshold_rule':'validation F1; ties choose higher threshold',
 'selection_metric':'validation F1','val_auc':float(metrics_f1['roc_auc']),
 'n_parameters':int(model.coef_.size+1),'grid_searched':None,'tie_with_default':None,
 'front_end_hash':front_end_hash,
 'scaler_fit_rows':int(fit_rows.sum()),
 'scaler_excluded_rows':int(stats_exclude.sum()),
 'condition_number_train_design':condition_number,
}
card_path=CARD_DIR / '10_linear_regression.json'
card_path.write_text(json.dumps(card,indent=2,allow_nan=False)+'\n',encoding='utf-8')
print('Frozen model:',model_path)
print('Verdict card:',card_path)
print('Decision log draft: Regress the binary label with balanced sample weights; fit the scaler on train rows excluding silent and near_silent only; choose the operating threshold by validation F1 with ties to the higher threshold; report ROC-AUC and miss rates at 1%/5% false-alarm caps.')

Frozen model:

 D:\Work\ML Course Project\Functional Project\results\models\10_linear_regression.joblib
Verdict card: D:\Work\ML Course Project\Functional Project\results\cards\10_linear_regression.json
Decision log draft: Regress the binary label with balanced sample weights; fit the scaler on train rows excluding silent and near_silent only; choose the operating threshold by validation F1 with ties to the higher threshold; report ROC-AUC and miss rates at 1%/5% false-alarm caps.
